# 05｜重复预测与 NMS：怎样从许多框中保留代表结果

上一课看到，一对多目标分配会让多个预测槽位同时学习同一个真实物体。训练成功以后，这些槽位都可能对同一物体给出高分且相近的边界框。

因此，模型的原始输出不一定等于最终展示给用户的检测结果。

本课只学习传统检测器常用的 NMS：它为什么出现，怎样逐轮删除重复框，以及它与置信度筛选、目标分配有什么不同。

## 1. 为什么同一个物体会得到多个预测框

密集检测器会在许多特征图位置和尺度上产生预测。一个真实物体附近的多个预测槽位都可能看到相似的视觉信息。

如果训练时一个真实目标被分配给多个正样本，那么这些正样本都会学习相同的类别和相近的真实框：

$$
\text{一只真实的狗}
\rightarrow\text{多个正样本}
\rightarrow\text{多个高分狗框}
$$

所以重复框不是偶然出现的无意义噪声，而是密集预测和一对多监督共同产生的自然结果。

## 2. 原始输出与最终检测结果不是一回事

检测模型可能输出成百上千条候选结果，其中包含：

- 真正物体的高质量预测。
- 同一物体的多个重复预测。
- 分数很低的不确定预测。
- 把背景误认为物体的错误预测。

传统检测推理流程通常还需要整理：

$$
\text{模型原始预测}
\rightarrow\text{低分筛选}
\rightarrow\operatorname{NMS}
\rightarrow\text{最终检测结果}
$$

NMS 处理的主要是同类别、高度重叠的重复框。

## 3. 置信度筛选和 NMS 解决不同问题

### 置信度或检测分数筛选

先删除分数低于阈值的预测，解决的是“模型自己都不太相信的结果是否保留”。

### NMS

在剩余较可信的预测中，继续检查哪些框可能重复描述同一个物体。

| 操作 | 主要依据 | 主要问题 |
|---|---|---|
| 分数筛选 | 单个预测的检测分数 | 这条预测是否足够可信 |
| NMS | 两个预测框的 IoU 与分数排序 | 两条可信预测是否重复 |

一个重复框可能分数很高，因此只做分数筛选并不能代替 NMS。

## 4. NMS 是什么

NMS 的全称是 Non-Maximum Suppression，中文常译为非极大值抑制。

名字可以拆成两部分：

- Maximum：优先保留当前分数最高的预测框。
- Suppression：抑制与它高度重叠的较低分框。

它的核心思想是：

> 如果多个同类别框高度重叠，就把它们暂时看成同一个物体的重复预测，保留分数最高的代表框，删除其余框。

## 5. 建立一个完整例子

假设图片中实际有两只狗和一只猫，模型输出了 6 个候选框：

| 框 | 预测类别 | 检测分数 | 直观情况 |
|---:|---|---:|---|
| $A$ | 狗 | 0.95 | 第一只狗的高质量框 |
| $B$ | 狗 | 0.88 | 与 $A$ 高度重叠的重复框 |
| $C$ | 狗 | 0.80 | 第二只狗的高质量框 |
| $D$ | 狗 | 0.63 | 与 $C$ 高度重叠的重复框 |
| $E$ | 猫 | 0.92 | 猫的高质量框 |
| $F$ | 狗 | 0.18 | 模型很不确定的低分框 |

现在设置：

$$
\text{分数阈值}=0.25,
\qquad\text{NMS IoU 阈值}=0.50
$$

## 6. 第一步：先删除低分框

框 $F$ 的检测分数为 0.18，低于分数阈值 0.25，因此先删除：

$$
0.18<0.25
$$

剩余候选为：

$$
A,B,C,D,E
$$

这一步只看每个框自己的分数，不比较框与框之间的 IoU，所以还没有开始 NMS 的重复判断。

## 7. 第二步：通常按类别分别处理

经典的按类别 NMS 会先把预测分组：

$$
\begin{aligned}
\text{狗类别}&:A,B,C,D\\
\text{猫类别}&:E
\end{aligned}
$$

这样做的原因是，不同类别的物体可能真实地占据相近位置。例如一个人可以骑在自行车上，两者的框可能高度重叠，但不能因为重叠就删除其中一个。

某些系统也支持类别无关 NMS，但当前先学习最常见、最容易理解的按类别版本。

## 8. 第三步：狗类别先按分数排序

狗类别候选按照分数从高到低排列：

$$
A(0.95)>B(0.88)>C(0.80)>D(0.63)
$$

NMS 每一轮都从尚未处理的框中选出当前最高分框，把它正式保留下来，再用它与其余同类别框比较 IoU。

因此第一轮先保留 $A$。

## 9. 第四步：用框 A 抑制重复框

假设框 $A$ 与其他狗框的 IoU 为：

| 比较 | IoU | 是否超过 0.50 | 处理结果 |
|---|---:|---:|---|
| $A$ 与 $B$ | 0.76 | 是 | 删除 $B$ |
| $A$ 与 $C$ | 0.12 | 否 | 保留 $C$ 继续参加后续轮次 |
| $A$ 与 $D$ | 0.08 | 否 | 保留 $D$ 继续参加后续轮次 |

因为：

$$
\operatorname{IoU}(A,B)=0.76>0.50
$$

$B$ 与高分框 $A$ 高度重叠且类别相同，所以被视为第一只狗的重复预测并删除。

## 10. 第五步：从剩余框继续下一轮

删除 $B$ 后，尚未处理的狗框为：

$$
C(0.80),D(0.63)
$$

当前最高分是 $C$，所以保留 $C$。再假设：

$$
\operatorname{IoU}(C,D)=0.67>0.50
$$

于是 $D$ 被视为第二只狗的重复框并删除。

狗类别最终保留：

$$
A,C
$$

## 11. 猫框为什么不会被狗框删除

猫类别只有框 $E$，因此直接保留。

即使 $E$ 与狗框 $A$ 在图像中高度重叠，按类别 NMS 也不会让狗框删除猫框，因为它们属于不同预测类别。

最后得到：

| 最终框 | 类别 | 分数 | 代表的物体 |
|---:|---|---:|---|
| $A$ | 狗 | 0.95 | 第一只狗 |
| $C$ | 狗 | 0.80 | 第二只狗 |
| $E$ | 猫 | 0.92 | 猫 |

6 个原始候选经过低分筛选与 NMS 后，整理成 3 条最终检测结果。

## 12. 把 NMS 算法写成完整流程

对于同一类别的一组预测框：

$$
\begin{aligned}
&1.\ \text{按检测分数从高到低排序}\\
&2.\ \text{取出当前最高分框并保留}\\
&3.\ \text{计算它与其余框的 IoU}\\
&4.\ \text{删除 IoU 超过阈值的较低分框}\\
&5.\ \text{对剩余框重复以上步骤}
\end{aligned}
$$

直到候选列表为空，所有已经选出的框组成这一类别的最终结果。

## 13. NMS 的 IoU 阈值控制什么

设 NMS 阈值为 $t_{nms}$。当较低分框与当前保留框满足：

$$
\operatorname{IoU}>t_{nms}
$$

它会被抑制。

### 阈值较低

只要有一定重叠就容易删除，去重更强，但可能误删靠得很近的两个真实物体。

### 阈值较高

只有高度重叠才删除，更容易保留相邻物体，但也可能留下重复框。

所以 NMS 阈值是在“删除重复框”和“保留相邻真实物体”之间做权衡。

## 14. 分数阈值与 NMS 阈值不能混淆

| 阈值 | 比较对象 | 调低后的主要影响 |
|---|---|---|
| 分数阈值 | 单个预测分数 | 保留更多低分候选，召回可能增加，误报和计算量也可能增加 |
| NMS IoU 阈值 | 两个同类别框的重叠度 | 更容易把相互重叠的框判为重复并删除 |

分数阈值回答“这条预测本身可信不可信”；NMS 阈值回答“这两条预测是否重复”。

它们虽然都出现在推理筛选阶段，但数值含义和作用完全不同。

## 15. NMS 为什么依赖分数排序

当两个框被认为重复时，NMS 必须决定保留谁。经典做法是保留检测分数更高的那个。

这隐含了一个假设：模型给出的分数越高，该预测越值得信任。

但分数高并不保证边界框一定更精确。如果分类分数与定位质量不一致，NMS 可能保留一个类别信心很高、定位却稍差的框。

因此，现代检测器也会研究怎样让检测分数更好地反映定位质量。不过当前不展开这些改进。

## 16. NMS 的局限：拥挤场景

如果两个人站得非常近，他们各自的真实边界框可能本来就高度重叠。

NMS 只看到：

- 两个框类别相同。
- IoU 很高。
- 其中一个分数较低。

它无法真正理解这究竟是同一个人的重复预测，还是两个靠得很近的人。因此可能把一个真实物体误删。

这说明 NMS 是有效的几何启发式后处理，但不是对场景内容的完整理解。

## 17. Hard NMS 与 Soft-NMS

本课讲的是经典 Hard NMS：只要重叠超过阈值，就直接删除较低分框。

Soft-NMS 的方向不同：它不一定立即删除重叠框，而是根据重叠程度降低它的分数。这样在拥挤场景中可能更温和。

现在只需要知道存在这种改进方向，不需要学习具体衰减公式。掌握经典 NMS 才是当前目标。

## 18. NMS 与目标分配怎样前后呼应

两者位于不同阶段：

$$
\begin{aligned}
\text{训练阶段}&:\text{一对多目标分配}\rightarrow\text{多个槽位学习同一物体}\\
\text{推理阶段}&:\text{多个槽位预测同一物体}\rightarrow\operatorname{NMS}\text{ 去重}
\end{aligned}
$$

目标分配需要真实标注，用于建立训练责任；NMS 不需要真实标注，只根据模型自己的类别、分数和预测框整理结果。

NMS 不会重新训练模型，也不产生反向传播。

## 19. DETR 为什么强调不需要 NMS

DETR 使用一对一集合匹配训练：每个真实物体只分配给一个 Query，未匹配的 Queries 学习 no object。

它希望模型直接学会输出一组无重复的最终目标：

$$
\text{一对一训练}
\rightarrow\text{每个目标一个代表预测}
\rightarrow\text{推理时不依赖 NMS}
$$

原始 DETR 推理时仍会根据类别分数去掉 no object 或低分预测，但不需要再使用经典 NMS 删除重复框。

这就是理解传统 NMS 后再学习 DETR 的价值：我们能看清 DETR 究竟删除了检测流程中的哪一个环节。

## 20. 常见误区

1. NMS 不是训练损失，也不会参与反向传播。
2. NMS 不需要真实框，它只处理模型预测。
3. 分数筛选不能代替 NMS，高分框之间仍可能重复。
4. NMS 通常删除的是同类别高重叠框，不是所有相互重叠的框。
5. IoU 高不一定代表同一个真实物体，拥挤场景可能出现误删。
6. DETR 不需要 NMS，不代表它不需要对 no object 和低分结果进行筛选。

## 21. 本节小结

这一课需要真正记住七个结论：

1. 一对多训练和密集预测会自然产生同一物体的多个候选框。
2. 分数筛选先删除不可信预测，NMS 再处理可信预测之间的重复。
3. NMS 每轮保留最高分框，并抑制与它同类别、高 IoU 的较低分框。
4. NMS IoU 阈值越低，去重越强，但越容易误删相邻物体。
5. NMS 是推理后处理，不需要真实标注，也不参与梯度计算。
6. NMS 在拥挤场景中可能无法区分重复框和相邻真实物体。
7. DETR 通过一对一集合匹配学习无重复输出，因此不依赖经典 NMS。

完整传统推理路线是：

$$
\text{大量原始预测}
\rightarrow\text{分数筛选}
\rightarrow\text{按类别执行 NMS}
\rightarrow\text{最终检测集合}
$$

下一课将学习 Precision、Recall、AP 和 mAP，回答“整理完预测以后，怎样评价整个检测器好不好”。

## 22. 自测问题

1. 为什么密集检测器容易为同一物体产生多个预测框？
2. 分数筛选与 NMS 分别解决什么问题？
3. NMS 为什么要先按检测分数排序？
4. 当前最高分框选出以后，要与哪些框计算 IoU？
5. 在教学例子中，框 $B$ 为什么被删除？
6. 框 $C$ 为什么没有被框 $A$ 删除？
7. 猫框 $E$ 为什么不会被狗框 $A$ 删除？
8. NMS 阈值降低后，删除行为怎样变化？
9. 分数阈值和 NMS IoU 阈值的数值含义有什么不同？
10. NMS 是否需要真实类别和真实框？
11. NMS 是否参与反向传播？
12. NMS 在拥挤场景中可能出现什么问题？
13. Soft-NMS 与 Hard NMS 的主要方向差异是什么？
14. DETR 为什么原则上不需要经典 NMS？

### 自测参考答案

1. 多个空间位置或尺度会产生候选，而且一对多分配可能让多个正样本学习同一个真实目标。
2. 分数筛选删除单独看不可信的预测；NMS 删除可信预测中的重复框。
3. 当两个框被视为重复时，需要优先保留模型更信任的那个。
4. 与尚未处理的同类别较低分框比较。
5. 它与更高分的同类别框 $A$ 的 IoU 为 0.76，超过 NMS 阈值 0.50。
6. 它与 $A$ 的 IoU 只有 0.12，可能代表另一只狗。
7. 经典按类别 NMS 分别处理不同类别。
8. 更小的重叠就可能触发删除，去重更强，也更容易误删相邻物体。
9. 分数阈值判断单个预测是否可信；NMS 阈值判断两个框是否过度重叠。
10. 不需要，只使用模型预测的类别、分数和边界框。
11. 不参与，它是推理阶段的后处理算法。
12. 可能把两个靠得很近的真实同类物体误认为重复预测。
13. Hard NMS 直接删除重叠框；Soft-NMS 更倾向于降低其分数。
14. 一对一集合匹配训练模型为每个真实目标产生一个代表预测，未匹配 Queries 学习 no object。